# Lab 07: EDA, Descriptive Statistics และ Statistical Inference
> CLO2 | LLo: คำนวณ descriptive statistics, สร้าง Confidence Interval และทดสอบสมมติฐานเบื้องต้นได้

**วิชา**: 1145 201 คณิตศาสตร์สำหรับวิทยาการข้อมูล | **Reference**: ISLP Ch.2.3 + Ch.3.1.2

---

## บทนำ

สัปดาห์นี้เราจะเรียนรู้ทักษะที่ Data Scientist ใช้ในวันแรกของทุกโปรเจกต์ นั่นคือ **Exploratory Data Analysis (EDA)** และ **Statistical Inference** ซึ่งเป็นสะพานเชื่อมระหว่าง data ดิบกับ insight ที่นำไปตัดสินใจได้ ก่อนที่เราจะสร้าง regression model ใน Week 8–9 เราต้องเข้าใจ distribution ของข้อมูลและความสัมพันธ์ระหว่างตัวแปรก่อน เป้าหมายของ Lab นี้คือให้นักศึกษาสามารถอ่านและตีความ `.describe()` correlation heatmap boxplot และผลจาก t-test ได้อย่างถูกต้อง สิ่งที่เรียนใน Lab นี้จะนำไปใช้โดยตรงใน Week 8 เมื่อเราตีความ SE t-statistic และ p-value ของ regression coefficients ทุก Data Scientist ที่ดีต้องสำรวจข้อมูลก่อนเสมอ — "Never model before you visualize" Lab นี้มี 3 TODOs: EDA บน Advertising dataset, Confidence Interval, และ Hypothesis Testing รวมถึง Case Study การวิเคราะห์ข้อมูลเศรษฐกิจประเทศไทย

In [ ]:
# ─── Import libraries ────────────────────────────────────────────
# วัตถุประสงค์: โหลด library ที่จำเป็นสำหรับ EDA และ Statistical Inference
# pandas: จัดการ tabular data
# matplotlib/seaborn: visualization สำหรับ EDA
# scipy.stats: สำหรับ CI และ hypothesis tests

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# ตั้งค่า style สำหรับ seaborn เพื่อให้กราฟสวยงาม
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

print('Libraries loaded!')
print('pandas:', pd.__version__, '| seaborn:', sns.__version__)

## Part 1: โหลดข้อมูลและ Descriptive Statistics

**Part นี้เราจะโหลด Advertising dataset และคำนวณ descriptive statistics เพื่อเข้าใจ distribution ของแต่ละตัวแปรก่อนสร้างโมเดล** — ขั้นตอนนี้คือ "sanity check" ที่ต้องทำเสมอ เพราะ model ที่ดีต้องเริ่มจากการเข้าใจข้อมูลก่อน

**Advertising dataset**: 200 ตลาด, 4 ตัวแปร
- `TV`, `Radio`, `Newspaper`: งบโฆษณา (พัน$) ใน 3 ช่องทาง
- `Sales`: ยอดขาย (พันหน่วย) — นี่คือ target variable ที่เราจะ predict ใน Week 8

In [ ]:
# ─── โหลด Advertising dataset ──────────────────────────────────────
# วัตถุประสงค์: ใช้ dataset นี้เพื่อฝึก EDA ก่อน regression ใน Week 8
# ถ้าไม่มีไฟล์ จะสร้าง dataset จำลองที่มีโครงสร้างเหมือนกัน

try:
    df = pd.read_csv('https://www.statlearning.com/s/Advertising.csv', index_col=0)
    print('โหลดจาก URL สำเร็จ')
except:
    # สร้าง dataset จำลองถ้า URL ไม่ work
    np.random.seed(42)
    n = 200
    tv = np.random.uniform(0.7, 296.4, n)
    radio = np.random.uniform(0.0, 49.6, n)
    newspaper = np.random.uniform(0.3, 114.0, n)
    sales = 7.03 + 0.0475*tv + 0.188*radio + 0.001*newspaper + np.random.normal(0, 1.5, n)
    df = pd.DataFrame({'TV': tv, 'Radio': radio, 'Newspaper': newspaper, 'Sales': sales})
    print('ใช้ dataset จำลอง')

print(f'Shape: {df.shape}')
print('\n5 แถวแรก:')
df.head()

In [ ]:
# ─── Descriptive Statistics ─────────────────────────────────────────
# วัตถุประสงค์: สรุป central tendency และ dispersion ของทุก column
# เพื่อตรวจสอบ range, outlier potential และ skewness เบื้องต้น

print('=== .describe() — สถิติพื้นฐาน ===')
print(df.describe().round(2))

print('\n=== ข้อมูลเพิ่มเติม ===')
print('Skewness (ความเบ้):') 
print(df.skew().round(3))
print('\nKurtosis (ความโด่ง):')
print(df.kurtosis().round(3))

In [ ]:
# ─── ตรวจสอบ Missing Values และ Data Types ───────────────────────────
# วัตถุประสงค์: ตรวจสอบว่ามี missing values หรือ wrong data types ก่อน analysis

print('=== .info() ===')
df.info()
print('\n=== Missing values ===')
print(df.isnull().sum())

### 🎯 TODO 1: EDA Visualization บน Advertising Dataset (ระดับ: Easy)

**เราต้องการ visualize distribution ของทุกตัวแปรและความสัมพันธ์ระหว่างกัน** เพื่อตัดสินใจว่าตัวแปรใดน่าจะเป็น predictor ที่ดีก่อนที่เราจะสร้าง regression model ใน Week 8 — "a picture is worth a thousand numbers"

ให้คุณ:
1. สร้าง **histogram** ของทุก 4 columns ในรูปเดียว (2×2 subplots) พร้อม title ชื่อตัวแปร
2. สร้าง **boxplot** เปรียบเทียบทั้ง 4 columns (ใช้ `df.boxplot()` หรือ `seaborn.boxplot`)
3. สร้าง **correlation heatmap** ด้วย `seaborn.heatmap(df.corr(), annot=True, cmap='coolwarm')`
4. สร้าง **scatter plot** ระหว่าง TV กับ Sales พร้อม regression line (`sns.regplot`)
5. **ตีความ**: จาก heatmap ตัวแปรใดมี correlation กับ Sales สูงสุด? ทำไม?

In [ ]:
# TODO 1: EDA Visualization
# Hint 1: fig, axes = plt.subplots(2, 2, figsize=(12, 8)) สำหรับ histogram grid
# Hint 2: สำหรับแต่ละ ax และ col: ax.hist(df[col], bins=20, edgecolor='black')
# Hint 3: df.corr() คืน correlation matrix, ใส่ใน sns.heatmap(..., vmin=-1, vmax=1)
# Hint 4: sns.regplot(data=df, x='TV', y='Sales') วาด scatter + regression line อัตโนมัติ

raise NotImplementedError('กรุณาเติม code ใน TODO 1')

## Part 2: Confidence Interval

**Part นี้เราจะคำนวณ Confidence Interval (CI) สำหรับ mean ของ Sales เพื่อเข้าใจว่าเราเชื่อมั่นในการประมาณ population mean ได้แค่ไหน** — CI ใน regression (Week 8–9) ใช้ logic เดียวกันนี้ทุกประการ เพียงแต่ใช้กับ coefficient estimates แทน

**95% CI สำหรับ mean**: $\bar{X} \pm t^* \cdot \frac{s}{\sqrt{n}}$

In [ ]:
# ─── Demo: คำนวณ 95% CI สำหรับ mean ของ Sales ────────────────────────
# วัตถุประสงค์: แสดงสูตร CI จาก scratch และเปรียบเทียบกับ scipy.stats

sales = df['Sales'].values
n = len(sales)
x_bar = sales.mean()
s = sales.std(ddof=1)   # sample std (ddof=1 สำหรับ unbiased)
se = s / np.sqrt(n)     # Standard Error

print(f'n = {n}')
print(f'x̄ = {x_bar:.4f}')
print(f's = {s:.4f}')
print(f'SE = s/√n = {se:.4f}')

# วิธีที่ 1: คำนวณจาก scratch ด้วย t-distribution
alpha = 0.05
t_star = stats.t.ppf(1 - alpha/2, df=n-1)  # t critical value
margin = t_star * se
ci_lower = x_bar - margin
ci_upper = x_bar + margin
print(f'\nt* = {t_star:.4f}')
print(f'95% CI (manual): ({ci_lower:.4f}, {ci_upper:.4f})')

# วิธีที่ 2: scipy.stats.t.interval
ci_scipy = stats.t.interval(0.95, df=n-1, loc=x_bar, scale=se)
print(f'95% CI (scipy):  {ci_scipy}')
print(f'ตรงกันไหม: {np.allclose(ci_lower, ci_scipy[0])}')

### 🎯 TODO 2: Bootstrap Confidence Interval (ระดับ: Medium)

**เราต้องการสร้าง Bootstrap CI สำหรับ correlation ระหว่าง TV และ Sales** เพราะ correlation ไม่มีสูตร CI ง่าย ๆ แบบ mean — Bootstrap ช่วยให้เราประมาณ CI สำหรับ statistic ใดก็ได้โดยไม่ต้องรู้ distribution ล่วงหน้า (เราจะเรียน Bootstrap อีกครั้งใน Week 14)

ให้คุณ:
1. คำนวณ Pearson correlation ระหว่าง TV และ Sales (ค่าจริง)
2. สร้าง Bootstrap samples จำนวน **B = 1000** รอบ: sample with replacement ขนาด n, คำนวณ correlation แต่ละรอบ
3. สร้าง **histogram** ของ bootstrap correlations
4. คำนวณ 95% Bootstrap CI จาก percentile: `np.percentile(boot_corrs, [2.5, 97.5])`
5. Plot: histogram + vertical lines แสดง CI bounds และ observed correlation

**ผลลัพธ์ที่คาดหวัง**: correlation TV–Sales ≈ 0.78, 95% CI ≈ (0.72, 0.84)

In [ ]:
# TODO 2: Bootstrap CI สำหรับ Correlation
# Hint: observed_corr = df['TV'].corr(df['Sales'])
# Hint: สำหรับแต่ละ bootstrap: idx = np.random.choice(n, size=n, replace=True)
#        boot_df = df.iloc[idx]; corr = boot_df['TV'].corr(boot_df['Sales'])
# Hint: boot_corrs = np.array([...]) เก็บ B correlations
# Hint: ci = np.percentile(boot_corrs, [2.5, 97.5])

np.random.seed(42)
B = 1000  # จำนวน bootstrap samples
n = len(df)

raise NotImplementedError('กรุณาเติม code ใน TODO 2')

## Part 3: Hypothesis Testing — t-test

**Part นี้เราจะทดสอบสมมติฐานเพื่อตอบคำถามเชิง statistical ว่ากลุ่มสองกลุ่มมี mean ต่างกันอย่างมีนัยสำคัญหรือไม่** — ทักษะนี้ใช้โดยตรงใน regression เมื่อ ISLP ทดสอบว่า H₀: β = 0 (predictor ไม่มีผลต่อ Y)

**Framework ของ Hypothesis Test**:
- H₀: ไม่มีความแตกต่าง (β = 0 หรือ μ₁ = μ₂)
- H₁: มีความแตกต่าง
- t-statistic = (estimate − H₀ value) / SE
- p-value = P(|T| ≥ |t_obs| | H₀ true)

In [ ]:
# ─── Demo: One-sample t-test ─────────────────────────────────────────
# วัตถุประสงค์: ทดสอบว่า mean ของ Sales แตกต่างจาก 14 หรือไม่
# H₀: μ = 14 (กำหนด null hypothesis)
# H₁: μ ≠ 14 (two-sided)

mu0 = 14.0
sales = df['Sales'].values

# คำนวณ t-statistic จาก scratch
n = len(sales)
x_bar = sales.mean()
s = sales.std(ddof=1)
se = s / np.sqrt(n)
t_stat = (x_bar - mu0) / se
p_value_manual = 2 * stats.t.sf(abs(t_stat), df=n-1)  # two-sided

print('=== One-sample t-test: H₀: μ = 14 ===')
print(f'x̄ = {x_bar:.4f}')
print(f't-statistic = {t_stat:.4f}')
print(f'p-value (manual) = {p_value_manual:.4f}')

# ใช้ scipy
t_scipy, p_scipy = stats.ttest_1samp(sales, popmean=mu0)
print(f'\nt-statistic (scipy) = {t_scipy:.4f}')
print(f'p-value (scipy)     = {p_scipy:.4f}')

alpha = 0.05
if p_scipy < alpha:
    print(f'\nสรุป: p = {p_scipy:.4f} < α = {alpha} → Reject H₀ → mean ≠ 14')
else:
    print(f'\nสรุป: p = {p_scipy:.4f} ≥ α = {alpha} → Fail to reject H₀')

### 🎯 TODO 3: Two-Sample t-test และ Chi-Square Test (ระดับ: Hard)

**เราต้องการทดสอบว่าตลาดที่มีงบ TV สูง (high_TV) และต่ำ (low_TV) มียอดขายเฉลี่ยต่างกันอย่างมีนัยสำคัญหรือไม่** การทดสอบนี้เหมือนกับที่ใช้ใน A/B testing ใน business

**ขั้นตอน**:
1. แบ่งกลุ่ม: `high_TV` = ตลาดที่ TV > median, `low_TV` = ตลาดที่ TV ≤ median
2. ทำ **Two-sample t-test**: H₀: μ_high = μ_low, H₁: μ_high > μ_low (one-sided)
   - ใช้ `stats.ttest_ind(high_TV_sales, low_TV_sales, alternative='greater')`
3. สร้าง **boxplot** เปรียบเทียบ Sales ของ 2 กลุ่ม
4. สร้าง **categorical variable** `TV_level` = 'High' ถ้า TV > median, ไม่งั้น 'Low'
   และ `Sales_level` = 'High' ถ้า Sales > median, ไม่งั้น 'Low'
5. สร้าง **contingency table** ด้วย `pd.crosstab(df['TV_level'], df['Sales_level'])`
6. ทำ **Chi-square test** ด้วย `stats.chi2_contingency(contingency)` — H₀: ไม่มีความสัมพันธ์
7. **ตีความ**: p-value ของ t-test และ chi-square บอกอะไร? สอดคล้องกับ correlation heatmap ไหม?

In [ ]:
# TODO 3: Two-sample t-test + Chi-square test
# Hint: median_tv = df['TV'].median()
# Hint: high_mask = df['TV'] > median_tv
# Hint: high_sales = df.loc[high_mask, 'Sales'].values
# Hint: t_stat, p_val = stats.ttest_ind(high_sales, low_sales, alternative='greater')
# Hint: chi2, p_chi2, dof, expected = stats.chi2_contingency(contingency)

raise NotImplementedError('กรุณาเติม code ใน TODO 3')

## Part 4: Case Study — EDA บนข้อมูล GDP ประเทศในเอเชียตะวันออกเฉียงใต้

**Part นี้เราจะทำ EDA แบบ end-to-end บนข้อมูลเศรษฐกิจจริงของประเทศในภูมิภาคเดียวกับเรา เพื่อฝึกกระบวนการวิเคราะห์ข้อมูลจริงที่ Data Scientist ใช้ทุกวัน**

**Case Study: GDP per capita (USD) ของประเทศในเอเชียตะวันออกเฉียงใต้ ปี 2022**

**Scenario**: สมมติว่าคุณเป็น economic analyst ที่ต้องรายงานความเหลื่อมล้ำทางเศรษฐกิจในภูมิภาค

In [ ]:
# ─── Case Study: GDP per capita ของ ASEAN ─────────────────────────────
# วัตถุประสงค์: ทำ full EDA pipeline บน real-world economic data
# เพื่อฝึก descriptive stats + CI + visualization ในบริบทจริง

# ข้อมูล GDP per capita (USD) และ life expectancy (years) ปี 2022
# Source: World Bank (approximate values)
asean_data = {
    'Country': ['Singapore', 'Brunei', 'Malaysia', 'Thailand', 'Indonesia',
                'Philippines', 'Vietnam', 'Myanmar', 'Cambodia', 'Laos'],
    'GDP_per_capita': [82808, 31087, 12364, 7232, 4788, 3623, 3756, 1210, 1593, 2535],
    'Life_expectancy': [83.5, 74.8, 76.4, 77.8, 71.7, 71.2, 73.7, 67.1, 69.8, 67.9],
    'Population_M': [5.9, 0.4, 33.6, 71.7, 277.5, 115.6, 98.2, 54.4, 17.7, 7.4]
}
df_asean = pd.DataFrame(asean_data)

print('=== ASEAN GDP Dataset ===')
print(df_asean.to_string(index=False))
print('\n=== Descriptive Statistics ===')
print(df_asean.describe().round(2))

# คำนวณ statistics เพิ่มเติม
gdp = df_asean['GDP_per_capita']
print(f'\nMedian GDP: ${gdp.median():,.0f}')
print(f'IQR: ${gdp.quantile(0.75) - gdp.quantile(0.25):,.0f}')
print(f'Skewness: {gdp.skew():.3f}  (positive → right-skewed → outlier ด้านบน)')

# 95% CI สำหรับ mean GDP
n = len(gdp)
ci_gdp = stats.t.interval(0.95, df=n-1, loc=gdp.mean(), scale=stats.sem(gdp))
print(f'\n95% CI สำหรับ mean GDP: (${ci_gdp[0]:,.0f}, ${ci_gdp[1]:,.0f})')
print('(ตีความ: ถ้า sample ซ้ำ 100 ครั้ง, 95 ครั้งจะได้ interval ที่ครอบ true mean)')

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart GDP per capita
colors = ['#2196F3' if c == 'Thailand' else '#90CAF9' for c in df_asean['Country']]
axes[0].barh(df_asean['Country'], df_asean['GDP_per_capita'], color=colors)
axes[0].set_xlabel('GDP per Capita (USD)')
axes[0].set_title('GDP per Capita (ASEAN, 2022)\nสีเข้ม = Thailand')
axes[0].axvline(gdp.mean(), color='red', linestyle='--', label=f'Mean: ${gdp.mean():,.0f}')
axes[0].axvline(gdp.median(), color='orange', linestyle='--', label=f'Median: ${gdp.median():,.0f}')
axes[0].legend()

# Scatter: GDP vs Life Expectancy
axes[1].scatter(df_asean['GDP_per_capita'], df_asean['Life_expectancy'],
                s=df_asean['Population_M']*0.5, alpha=0.7, color='steelblue')
for _, row in df_asean.iterrows():
    axes[1].annotate(row['Country'], (row['GDP_per_capita'], row['Life_expectancy']),
                     fontsize=8, xytext=(5, 5), textcoords='offset points')
axes[1].set_xlabel('GDP per Capita (USD)')
axes[1].set_ylabel('Life Expectancy (years)')
axes[1].set_title('GDP vs Life Expectancy (size = population)')

plt.tight_layout()
plt.show()

# Correlation
r, p_corr = stats.pearsonr(df_asean['GDP_per_capita'], df_asean['Life_expectancy'])
print(f'\nCorrelation GDP–Life Expectancy: r = {r:.3f}, p-value = {p_corr:.3f}')
print('Insight: ประเทศที่ GDP per capita สูงกว่ามีอายุขัยยืนยาวกว่า')
print(f'Thailand GDP per capita = ${df_asean.loc[df_asean["Country"]=="Thailand", "GDP_per_capita"].values[0]:,}')
print(f'= อยู่ที่ percentile ที่ {stats.percentileofscore(gdp, 7232):.0f} ของภูมิภาค')

## Reflection Questions

**คำถามที่ 1**: ใน TODO 3 ถ้า p-value ของ t-test = 0.001 และ p-value ของ chi-square = 0.003 ควรสรุปว่าอย่างไร? และมีความแตกต่างอะไรระหว่าง "statistically significant" กับ "practically significant" (effect size)?

*(เขียนคำตอบที่นี่)*

---

**คำถามที่ 2**: จาก Case Study ASEAN GDP — GDP per capita ของไทยอยู่ที่ตำแหน่งใดของภูมิภาค? ถ้าคุณต้องรายงานให้ผู้บริหารเข้าใจ ควรใช้ mean หรือ median เป็นตัวแทน และทำไม? (Hint: right-skewed distribution)

*(เขียนคำตอบที่นี่)*

---

**คำถามที่ 3**: ใน regression ที่จะเรียนใน Week 8–9 t-statistic สำหรับ coefficient β̂₁ คำนวณเป็น t = β̂₁ / SE(β̂₁) ซึ่งเหมือนกับสูตรที่ใช้ใน one-sample t-test ทุกประการ อธิบายว่า H₀ ในกรณีนี้คืออะไร และ p-value < 0.05 หมายความว่าอะไรสำหรับ predictor นั้น

*(เขียนคำตอบที่นี่)*